In [29]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Étape 4 — Optimisation et évaluation.

1 - Imports et chargement des données

In [30]:
df = pd.read_csv("../data/raw/student_mental_health.csv")

X = df.drop(columns=["Student_ID", "Mental_Health_Risk"])
y = df["Mental_Health_Risk"]

X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

data = joblib.load(
    "../data/processed/preprocessed_data.joblib"
)

preprocessor = data["preprocessor"]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (1600, 15)
Testing: (400, 15)


2 - Reproducing the data cleaning

In [ ]:
# invalid_train = (
#     X_train["Social_Media_Hours"] > X_train["Screen_Time_Hours"]
# )

# invalid_test = (
#     X_test["Social_Media_Hours"] > X_test["Screen_Time_Hours"]
# )

# print("Training inconsistencies:", invalid_train.sum())
# print("Testing inconsistencies:", invalid_test.sum())

# X_train.loc[invalid_train, "Social_Media_Hours"] = np.nan
# X_test.loc[invalid_test, "Social_Media_Hours"] = np.nan

Training inconsistencies: 27
Testing inconsistencies: 10


3 - Création d'un pipeline pour chaque modèle

In [32]:
pipelines = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    random_state=42,
                ),
            ),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                RandomForestClassifier(
                    random_state=42,
                ),
            ),
        ]
    ),
    "KNN": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", KNeighborsClassifier()),
        ]
    ),
}

4 - Définition des hyperparamètres pour chaque modèle

In [33]:
param_grids = {
    "Logistic Regression": {"classifier__C": [0.01, 0.1, 1, 10]},
    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 10, 20],
        "classifier__min_samples_split": [2, 5],
    },
    "KNN": {
        "classifier__n_neighbors": [3, 5, 7, 9],
        "classifier__weights": ["uniform", "distance"],
    },
}

5 - Création de GridSearchCV

In [34]:
grid_searches = {}

for name , pipeline in pipelines.items():
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        cv=5,
        scoring="f1_weighted",
        n_jobs=-1,
        verbose=1,
    )

    grid_searches[name] = grid_search

# grid_searches


In [ ]:
for name , grid_search in grid_searches.items():

    print(f"\noptimizing: {name}")

    grid_search.fit(X_train , y_train)

    print("best parameters: ", grid_search.best_params_)
    print("best cross-validation F1: ", grid_search.best_score_)


optimizing: Logistic Regression
Fitting 5 folds for each of 4 candidates, totalling 20 fits


best parameters:  {'classifier__C': 1}
best cross-validation F1:  0.745898106327724

optimizing: Random Forest
Fitting 5 folds for each of 18 candidates, totalling 90 fits
best parameters:  {'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 300}
best cross-validation F1:  0.68276636295661

optimizing: KNN
Fitting 5 folds for each of 8 candidates, totalling 40 fits
best parameters:  {'classifier__n_neighbors': 7, 'classifier__weights': 'distance'}
best cross-validation F1:  0.6424110492377066


: 